# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to explore and process a dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, leveraging the dataset's Croissant schema for structured access and analysis.

### Dataset Source
The dataset is published as a FAIR Croissant package and described by a JSON-LD schema accessible via:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure the mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print high-level metadata
meta = dataset.metadata
print(f"{meta.name}:\n{meta.description}")

# Print further context
print(f"\nPublished: {meta.date_published}\nVersion: {meta.version}\nIdentifier: {meta.identifier}\nLicense: {meta.license}")

## 2. Data Overview
Explore the record sets and their fields to understand the data structure.

According to the schema, each entity (record set, field, and column) is uniquely referenced by its `@id`. We'll print out all available record sets and their fields' IDs.

In [ ]:
# List all available record sets with their @id and name
print("Record Sets (@id, name):")
record_sets = [rs for rs in getattr(meta, 'record_sets', [])]
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']}")
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"    - {field['@id']}: {field.get('name', '(no name)')}")
if not record_sets:
    print('(No record sets are directly listed in the metadata. Attempting to retrieve via dataset API...)')

# ---
# Try auto-discovering record sets from the Croissant schema via mlcroissant
print("\nDiscovered record sets via mlcroissant:")
discovered_record_sets = []
for rs in dataset.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")
    discovered_record_sets.append(rs['@id'])
    if 'fields' in rs:
        print("    Fields:")
        for f in rs['fields']:
            print(f"    - {f['@id']}: {f.get('name', '(no name)')}")
if not discovered_record_sets:
    print("No record sets found through the dataset. Please ensure the dataset has data.\n")
else:
    print(f"\nTotal record sets discovered: {len(discovered_record_sets)}")

## 3. Data Extraction
Load one or more record sets into Pandas DataFrames for analysis.

**Entities are referenced by their `@id` fields throughout.**

We'll load all available record sets.

In [ ]:
# List record set @ids for extraction
record_set_ids = discovered_record_sets
dataframes = {}

if not record_set_ids:
    print("No record set IDs found; cannot extract data.")
else:
    for rs_id in record_set_ids:
        # Load all records as a list of dicts
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded '{rs_id}' into DataFrame with shape: {df.shape}")
        print(f"  Columns (@id): {list(df.columns)}")
    # Show head for the first (or default) record set
    example_rs = record_set_ids[0]
    print("\nSample records from the first record set:")
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Process and explore the data: filter, normalize, group, and summarize. All operations should reference fields by their `@id`.

_Demo: Select a numeric field and group by a categorical field._

In [ ]:
# Select the first record set (replace this if you prefer another)
record_set_id = record_set_ids[0] if record_set_ids else None
if not record_set_id:
    print("No record set available for EDA.")
else:
    df = dataframes[record_set_id]
    print(f"Investigating record set: {record_set_id}")

    # Try to guess a numeric field
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric fields auto-detected in the DataFrame.")
    else:
        # Filter rows where the numeric value exceeds a threshold (e.g. its median or a constant)
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (using @id):")
        display(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
        print(f"\nAdded normalized column for {numeric_field_id} (referenced by @id):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to group by a categorical field
        group_field = None
        for col in df.columns:
            if col == numeric_field_id:
                continue
            if df[col].dtype == object and df[col].nunique() < 10:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped means of {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found in this record set.")

## 5. Visualization
Visualize distributions or relationships between fields, referencing columns and axes by their `@id` fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not dataframes.get(record_set_id) or numeric_field_id is None:
    print('No numeric data available for plotting.')
else:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id} (@id)')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} vs {group_field} (@id)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to:
- Load a dataset described by a Croissant JSON-LD schema,
- Discover available record sets and their fields via `@id`,
- Extract and process data in a structured manner,
- Apply basic filtering, normalization, and grouping using field `@id` references,
- Generate summary visualizations.

This workflow can be extended to more advanced analyses by utilizing further field metadata, relations, and Croissant-specific capabilities.

**For robust usage, always reference data entities by their `@id` as shown throughout this notebook.**